# Invariant engine pieces
- import setup
- load config
- load tabular input
- iterate row-wise
- build row payload
- call model
- parse structured JSON
- retry logic
- append output columns
- export results
- optional batch-level normalization hook

In [ ]:
import openai
import asyncio
import os
import pandas as pd  # For data handling, like reading from Excel
from openai import AsyncOpenAI  # Asynchronous client from the new OpenAI SDK
from rapidfuzz import fuzz
from dotenv import load_dotenv
import json
import configparser  # For reading configuration files
import re

import nest_asyncio
nest_asyncio.apply()

# Load .env variables
- Load required environment variables for API access and runtime configuration.  
- This includes credentials and identifiers used to initialize the model client and associated services.

In [ ]:
# Load .env variables

load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
assistant_id     = os.getenv("ASSISTANT_ID")
healmatch_id     = os.getenv("HEALMATCH_ID")
harmonizer_id    = os.getenv("HARMONIZER_ID")

# sanity check (optional)
if not all([openai_api_key, assistant_id, healmatch_id, harmonizer_id]):
    raise RuntimeError("Missing one or more OpenAI env vars")

client = AsyncOpenAI(api_key=openai_api_key)

# Load configuration file
- Load the workflow configuration file, which defines all user-editable parameters for the pipeline.  
- This includes input file paths, worksheet names, and the key columns used for row-level evaluation.
- Most modifications to the workflow should be made in this configuration file rather than the script.

In [ ]:
# Load configuration file
config = configparser.ConfigParser()
config.read('config_prestep.ini')

# Retrieve file paths and column names from config
## Modify below and in .config file for desired key input columns
input_file = config['Files']['input_file']
input_worksheet = config['Files']['input_worksheet']
A_column = config['Columns']['a_column']
B_column = config['Columns']['b_column']

## Optional: Initialize logging

Set up lightweight logging to capture key workflow events such as model responses, parsing issues, and export status.

Logging is optional but recommended for debugging and traceability.  
Logs will be written to the `logs/` folder.

In [ ]:
import logging
from pathlib import Path
from datetime import datetime

# Create logs directory if it doesn't exist
logs_dir = Path("logs")
logs_dir.mkdir(exist_ok=True)

# Create a simple timestamped log file
log_file = logs_dir / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger()

logger.info("Workflow started")

# Set AI prompt instructions
- Load the workflow configuration file, which defines all user-editable parameters for the pipeline.  
- This includes input file paths, worksheet names, and the key columns used for row-level evaluation.
- Most modifications to the workflow should be made in this configuration file rather than the script.

In [ ]:
# Set AI prompt instructions
prompt1_id = config['Instructions']['prompt1']
prompt2_id = config['Instructions']['prompt2']
prompt3_id = config['Instructions']['prompt3']

# Load data frame
- Load the input dataset and subset to the configured columns of interest.  
- These columns define the fields that will be used to construct row-level payloads for evaluation.

In [ ]:
# Load data frame
# Load the data dictionary from input file
data_dict_df = pd.read_excel(input_file, sheet_name=input_worksheet)

# Select only the relevant columns
data_dict_df = data_dict_df[[A_column, B_column]]

logger.info(f"Loaded data shape: {data_dict_df.shape}")

# Build row-level context / payload
Construct a structured payload for each row using the configured source columns.  
This payload is the unit passed into the evaluator.

In [ ]:
# Build row-level context / payload

def build_row_payload(row, config):
    return {
        "row_id": row.name,
        "source_fields": {
            A_column: row[A_column],
            B_column: row[B_column],
        }
    }

sample_payload = build_row_payload(data_dict_df.iloc[0], config)
sample_payload

# Run row-wise model evaluation
Before looping, you want a reusable function that:
- takes a single row payload
- injects it into your prompt
- calls the model
- returns structured output

Think of this as your “row interpreter”

In [ ]:
# Run row-wise model evaluation

async def evaluate_row(payload, client, config):
    # Extract prompt ID from config
    prompt_id = config['Instructions']['prompt1']

    # Build input (you can refine this later)
    input_text = f"""
    Variable: {payload['source_fields'][A_column]}
    Description: {payload['source_fields'][B_column]}
    """
    logger.info(f"Row {payload['row_id']} model response received")
    
    # Call model (adjust depending on your API style)
    response = await client.responses.create(
        model="gpt-4.1-mini",
        input=input_text
    )

    # TODO: parse structured output
    return response.output_text

# Run row-wise loop
- Process each row as an independent unit by constructing a payload and passing it into the evaluator.  
- This step applies the same configured logic across all rows, producing consistent, structured outputs for downstream review.

In [ ]:
# Run row-wise loop

async def run_evaluation(df, config):
    results = []

    for _, row in df.iterrows():
        payload = build_row_payload(row, config)

        result = await evaluate_row(payload, client, config)

        results.append({
            "row_id": payload["row_id"],
            **result
        })

    return results

results = asyncio.run(run_evaluation(data_dict_df, config))


# Parse structured outputs
- Parse each model response into a predictable set of structured fields for downstream review.  
- This step translates raw evaluator output into tabular values, while preserving the original response and handling malformed outputs consistently.

In [ ]:
def parse_structured_output(response_text):
    """
    Parse the model's response text into a flat structured dictionary
    that can be merged into the output table.
    """
    try:
        parsed = json.loads(response_text)

        return {
            "parsed_successfully": True,
            "output_label": parsed.get("output_label"),
            "output_confidence": parsed.get("output_confidence"),
            "output_rationale": parsed.get("output_rationale"),
            "raw_response": response_text
        }
    
    except Exception as e:
        logger.warning(f"Parsing failed: {str(e)}")

        return {
            "parsed_successfully": False,
            "output_label": None,
            "output_confidence": None,
            "output_rationale": f"Parsing error: {str(e)}",
            "raw_response": response_text
        }

# Apply parsing within evaluation loop
- Iterate over each row, build the payload, run the evaluator, and parse the resulting output.  
- This step combines row-wise evaluation and structured parsing to produce a unified results collection.

In [ ]:
# Apply parsing within evaluation loop

results = []

for _, row in data_dict_df.iterrows():
    payload = build_row_payload(row, config)
    
    result = await evaluate_row(payload, client, config)
    
    parsed_result = parse_structured_output(result)

    results.append({
        "row_id": payload["row_id"],
        **parsed_result
    })

# Convert results → structured table
- Convert the collected row-level results into a structured table and merge with the original dataset.  
- This step produces a unified, review-ready dataframe containing both source fields and model-derived outputs.

In [ ]:
# Convert results → structured table

results_df = pd.DataFrame(results)

# Optional but recommended: merge with original data
final_df = data_dict_df.merge(
    results_df,
    left_index=True,
    right_on="row_id",
    how="left"
)

final_df.head()

# Export results
- Write the final structured dataset to disk for downstream review and analysis.  
- The exported file contains both original input fields and parsed model outputs.

In [ ]:
# Export results

output_file = config['Files'].get('output_file', 'output.xlsx')

final_df.to_excel(output_file, index=False)

print(f"Results exported to {output_file}")

logger.info(f"Results exported to {output_file}")
logger.info("Workflow completed")

> ⚠️ Logging Tip  
> Consider logging parsing failures and model responses to help debug prompt or schema issues.